# Lesson 7.1 — Inverse Velocity Kinematics
**Module 6 · Unit 7 · Lesson 25**

To move the tool a desired way, solve ξ = J q̇ for q̇. Square non-singular case: **q̇ = J⁻¹ξ = VΣ⁻¹Uᵀξ** (un-stretching the ellipsoid).

In [1]:
import numpy as np
def dh(th,d,a,al):
    ct,st,ca,sa=np.cos(th),np.sin(th),np.cos(al),np.sin(al)
    return np.array([[ct,-st*ca,st*sa,a*ct],[st,ct*ca,-ct*sa,a*st],[0,sa,ca,d],[0,0,0,1]])
def forward_chain(P,T,q):
    M=np.eye(4); Ms=[M.copy()]
    for i,(th0,d0,a,al) in enumerate(P):
        th,d=(th0+q[i],d0) if T[i]=="R" else (th0,d0+q[i]); M=M@dh(th,d,a,al); Ms.append(M.copy())
    return Ms
def geometric_jacobian(P,T,q):
    Ms=forward_chain(P,T,q); on=Ms[-1][:3,3]; J=np.zeros((6,len(q)))
    for i in range(len(q)):
        z=Ms[i][:3,2]; o=Ms[i][:3,3]
        if T[i]=="R": J[:3,i]=np.cross(z,on-o); J[3:,i]=z
        else: J[:3,i]=z
    return J
def Jv_planar(P,T,q): return geometric_jacobian(P,T,q)[:2,:]
def fk_xy(P,T,q):
    M=forward_chain(P,T,q)[-1]; return M[:2,3].copy()
def dls_inv(J,lam): return J.T@np.linalg.inv(J@J.T+lam**2*np.eye(J.shape[0]))
P2=[(0,0,1,0),(0,0,1,0)]; T2=["R","R"]
P3=[(0,0,1,0),(0,0,1,0),(0,0,0.6,0)]; T3=["R","R","R"]


## Square inverse achieves the commanded twist (round trip)

In [2]:
checks=[]
J=Jv_planar(P2,T2,np.array([0.5,0.8])); xi=np.array([0.3,-0.2])
qd=np.linalg.solve(J,xi)
print("q_dot =",np.round(qd,4),"  J q_dot =",np.round(J@qd,4),"  (== xi)")
checks.append(np.allclose(J@qd,xi))

q_dot = [-0.1568 -0.0766]   J q_dot = [ 0.3 -0.2]   (== xi)


## J⁻¹ = V Σ⁻¹ Uᵀ matches the direct inverse

In [3]:
U,S,Vt=np.linalg.svd(J)
Jinv_svd=Vt.T@np.diag(1/S)@U.T
checks.append(np.allclose(Jinv_svd,np.linalg.inv(J),atol=1e-9))
print("V Sigma^-1 U^T == inv(J):",checks[-1])

V Sigma^-1 U^T == inv(J): True


## Conditioning degrades toward a singularity

In [4]:
for t2 in [1.2,0.5,0.1]:
    s=np.linalg.svd(Jv_planar(P2,T2,np.array([0.4,t2])),compute_uv=False)
    print(f"theta2={t2}: 1/sigma_min (worst joint-rate gain) = {1/s[-1]:.2f}")
checks.append(1/np.linalg.svd(Jv_planar(P2,T2,np.array([0.4,0.1])),compute_uv=False)[-1] > 1/np.linalg.svd(Jv_planar(P2,T2,np.array([0.4,1.2])),compute_uv=False)[-1])
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

theta2=1.2: 1/sigma_min (worst joint-rate gain) = 2.00
theta2=0.5: 1/sigma_min (worst joint-rate gain) = 4.53
theta2=0.1: 1/sigma_min (worst joint-rate gain) = 22.37
All checks passed.
